# Experiment: IFEval Selector Playground (OpenAI-Compatible Endpoint)

Use this notebook to test a rule selector against a **real IFEval sample** using your lab endpoint:
- Base URL pattern: `http://<host>:<port>/v1/chat/completions`
- Auth: `Bearer <token>`
- Model: OpenAI-compatible chat model

This notebook mirrors the selector request/response shape used by dynamic InstABoost.


## 1) Setup

This cell locates the repo root, imports IFEval helpers, and imports the selector parser used in the main codepath.


In [1]:
from __future__ import annotations

import json
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'datasets').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing src/ and datasets/.')


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dynamic_boost.selector_llm import build_selector_payload
from src.dynamic_boost.types import SelectorRequest
from src.ifeval_dynamic.data_adapter import clean_kwargs, load_ifeval_samples
from src.ifeval_dynamic.eval_adapter import _resolve_instruction_dict
from src.ifeval_dynamic.selector_context import build_selector_context

print(f'Repo root: {REPO_ROOT}')


Repo root: /Users/vitoriag/Documents/multi-rules


## 2) Endpoint + Sample Config

Set your endpoint and token here (or via environment variables).


In [2]:
SELECTOR_BASE_URL = os.getenv('SELECTOR_BASE_URL', 'http://158.130.111.238:8000')
SELECTOR_API_KEY = os.getenv('SELECTOR_API_KEY', 'brachiokey')
SELECTOR_MODEL = os.getenv('SELECTOR_MODEL', 'openai/gpt-oss-120b')
SELECTOR_TIMEOUT_S = float(os.getenv('SELECTOR_TIMEOUT_S', '60'))

SPLIT = 'val'  # 'val' or 'test'
SAMPLE_INDEX = 2
N_VAL = 100
N_TEST = 400
SEED = 42
MS_JSONL_PATH = None  # optional local path to ifeval_wo_instructions.jsonl

INITIAL_ACTIVE_IDS_MODE = 'first_only'  # 'first_only' or 'all'
MAX_GENERATION_CHARS = 1600

SELECTOR_SYSTEM_PROMPT = """
You are an instruction-selector for dynamic attention boosting.

Task:
Choose which instruction IDs should be emphasized for the NEXT generated tokens.

Constraints:
- Be instruction-agnostic: do not hardcode logic for any specific instruction ID.
- Use only the provided candidate IDs, instruction texts, current generation, and metadata.
- If metadata.instruction_progress_hints exists, treat it as advisory evidence.

Decision policy:
1) Prefer keeping/adding instructions that currently appear unsatisfied.
2) Prefer de-prioritizing instructions that appear satisfied, unless they still look at risk.
3) Keep the active set minimal but sufficient for the next tokens.

Output format:
Return ONLY valid JSON with keys: decision, active_instruction_ids, confidence, reason.
decision must be one of stay, switch, add.
active_instruction_ids must be a subset of candidate_instruction_ids (empty list is allowed).
confidence must be a float in [0,1].
""".strip()

print('Selector endpoint:', SELECTOR_BASE_URL)
print('Selector model:', SELECTOR_MODEL)
if not SELECTOR_API_KEY:
    print('Warning: SELECTOR_API_KEY is empty. Set it before calling the endpoint.')


Selector endpoint: http://158.130.111.238:8000
Selector model: openai/gpt-oss-120b


## 3) Load A Real IFEval Sample

This pulls the same normalized sample structure used by the runner.


In [3]:
val_samples, test_samples = load_ifeval_samples(
    n_val=N_VAL,
    n_test=N_TEST,
    seed=SEED,
    ms_jsonl_path=MS_JSONL_PATH,
)

samples = val_samples if SPLIT == 'val' else test_samples
if not samples:
    raise ValueError(f'No samples loaded for split={SPLIT!r}')

if SAMPLE_INDEX < 0 or SAMPLE_INDEX >= len(samples):
    raise IndexError(f'SAMPLE_INDEX={SAMPLE_INDEX} out of range [0, {len(samples)-1}]')

sample = samples[SAMPLE_INDEX]
if INITIAL_ACTIVE_IDS_MODE == 'all':
    initial_active_ids = list(sample.instruction_id_list)
else:
    initial_active_ids = [sample.instruction_id_list[0]]

print('Sample index:', SAMPLE_INDEX)
print('Sample ID:', sample.sample_id)
print('IFEval key:', sample.key)
print('Initial active IDs:', initial_active_ids)
print()
print('Base question:')
print(sample.base_question)
print()
print('Instructions:')
for i, (inst_id, inst_text, inst_kwargs) in enumerate(
    zip(sample.instruction_id_list, sample.instruction_texts, sample.kwargs_list),
    start=1,
):
    print(f'{i}. {inst_id}')
    print(f'   kwargs={inst_kwargs}')
    print(f'   text={inst_text}')


Sample index: 2
Sample ID: ifeval_2482
IFEval key: 2482
Initial active IDs: ['combination:repeat_prompt']

Base question:
Rewrite the following sentence in a style that is unusual: "But when the people of the land came to know that the Philistines had fled, they departed from Saul and went after David."

Instructions:
1. combination:repeat_prompt
   kwargs={'num_highlights': None, 'relation': None, 'num_words': None, 'num_placeholders': None, 'prompt_to_repeat': 'Rewrite the following sentence in a style that is unusual: "But when the people of the land came to know that the Philistines had fled, they departed from Saul and went after David."', 'num_bullets': None, 'section_spliter': None, 'num_sections': None, 'capital_relation': None, 'capital_frequency': None, 'keywords': None, 'num_paragraphs': None, 'language': None, 'let_relation': None, 'letter': None, 'let_frequency': None, 'end_phrase': None, 'forbidden_words': None, 'keyword': None, 'frequency': None, 'num_sentences': None, '

## 4) Build Selector Payload For Current Generation State

Edit `CURRENT_GENERATION` to test different partial outputs.


In [4]:
def resolve_instruction_dict_safely() -> tuple[dict[str, Any] | None, str | None]:
    try:
        return _resolve_instruction_dict(None), None
    except Exception as exc:  # noqa: BLE001
        return None, f'{type(exc).__name__}: {exc}'


INSTRUCTION_DICT, INSTRUCTION_DICT_ERROR = resolve_instruction_dict_safely()
if INSTRUCTION_DICT_ERROR:
    print('Warning: failed to load instruction checkers:', INSTRUCTION_DICT_ERROR)
    print('Progress hints will use unknown statuses.')


def build_instruction_progress_hints(sample_obj: Any, current_generation: str) -> dict[str, dict[str, Any]]:
    hints: dict[str, dict[str, Any]] = {}

    for inst_id, kwargs in zip(sample_obj.instruction_id_list, sample_obj.kwargs_list):
        hint: dict[str, Any] = {
            'status': 'unknown',
            'currently_passes': None,
            'switch_away_recommended': False,
        }

        if INSTRUCTION_DICT is None:
            hint['status'] = 'unknown_no_checker_backend'
            hints[inst_id] = hint
            continue

        if inst_id not in INSTRUCTION_DICT:
            hint['status'] = 'unknown_instruction_id'
            hints[inst_id] = hint
            continue

        try:
            checker = INSTRUCTION_DICT[inst_id](inst_id)
            safe_kwargs = clean_kwargs(kwargs if isinstance(kwargs, dict) else {})
            checker.build_description(**safe_kwargs)
            currently_passes = bool(checker.check_following(current_generation))

            hint['currently_passes'] = currently_passes
            hint['status'] = 'currently_passes' if currently_passes else 'currently_fails'
            hint['switch_away_recommended'] = bool(currently_passes)
        except Exception as exc:  # noqa: BLE001
            hint['status'] = 'checker_error'
            hint['checker_error'] = f'{type(exc).__name__}: {exc}'

        hints[inst_id] = hint

    return hints


CURRENT_GENERATION = ''
GENERATION_TOKEN_COUNT = 0
STEP_INDEX = 1

progress_hints = build_instruction_progress_hints(sample, CURRENT_GENERATION)

ctx = build_selector_context(
    sample=sample,
    current_generation=CURRENT_GENERATION,
    active_instruction_ids=initial_active_ids,
    generation_token_count=GENERATION_TOKEN_COUNT,
    step_index=STEP_INDEX,
    max_generation_chars=MAX_GENERATION_CHARS,
    extra_metadata={
        'kwargs_list': sample.kwargs_list,
        'instruction_progress_hints': progress_hints,
    },
)

request = SelectorRequest(
    sample_id=ctx['sample_id'],
    base_prompt=ctx['base_prompt'],
    candidate_instruction_ids=ctx['candidate_instruction_ids'],
    instruction_text_by_id=ctx['instruction_text_by_id'],
    currently_active_instruction_ids=ctx['currently_active_instruction_ids'],
    current_generation=ctx['current_generation'],
    generation_token_count=ctx['generation_token_count'],
    step_index=ctx['step_index'],
    metadata=ctx['metadata'],
)

print('Instruction progress statuses:')
for inst_id in request.candidate_instruction_ids:
    status = progress_hints.get(inst_id, {}).get('status', 'unknown')
    print(f'- {inst_id}: {status}')
print()

payload = build_selector_payload(request, max_generation_chars=MAX_GENERATION_CHARS)
payload_preview = json.dumps(payload, indent=2)
print(payload_preview[:4000])
if len(payload_preview) > 4000:
    print()
    print('... payload preview truncated ...')


Instruction progress statuses:
- combination:repeat_prompt: currently_fails

{
  "model_input": "Rewrite the following sentence in a style that is unusual: \"But when the people of the land came to know that the Philistines had fled, they departed from Saul and went after David.\"\n\nYour response should follow the instructions below:\n- First repeat the request word for word without change, then give your answer (1. do not say any words or characters before repeating the request; 2. the request you need to repeat does not include this sentence)",
  "step_index": 1,
  "generation_token_count": 0,
  "candidate_instruction_ids": [
    "combination:repeat_prompt"
  ],
  "instruction_texts": [
    {
      "id": "combination:repeat_prompt",
      "text": "First repeat the request word for word without change, then give your answer (1. do not say any words or characters before repeating the request; 2. the request you need to repeat does not include this sentence)"
    }
  ],
  "currently_ac

## 5) Call Your OpenAI-Compatible Endpoint

This sends the selector payload as the user message and expects JSON in `choices[0].message.content`.


In [5]:
def call_openai_chat_completion(
    *,
    payload: dict[str, Any],
    system_prompt: str = SELECTOR_SYSTEM_PROMPT,
) -> tuple[str, dict[str, Any]]:
    if not SELECTOR_API_KEY:
        raise ValueError('SELECTOR_API_KEY is empty. Set it before calling the endpoint.')

    url = SELECTOR_BASE_URL.rstrip('/') + '/v1/chat/completions'
    body = {
        'model': SELECTOR_MODEL,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': json.dumps(payload, ensure_ascii=True, indent=2)},
        ],
        'temperature': 0.0,
    }

    req = urllib.request.Request(
        url,
        data=json.dumps(body).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {SELECTOR_API_KEY}',
        },
        method='POST',
    )

    try:
        with urllib.request.urlopen(req, timeout=SELECTOR_TIMEOUT_S) as resp:
            response_json = json.loads(resp.read().decode('utf-8'))
    except urllib.error.HTTPError as exc:
        err_body = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'HTTP {exc.code} from selector endpoint: {err_body}') from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(f'Failed to reach selector endpoint: {exc}') from exc

    choices = response_json.get('choices')
    if not isinstance(choices, list) or not choices:
        raise RuntimeError("Response missing non-empty 'choices'.")

    message = choices[0].get('message')
    if not isinstance(message, dict):
        raise RuntimeError("Response missing 'choices[0].message'.")

    content = message.get('content')
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError('Selector response content is empty.')

    return content, response_json


raw_selector_text, raw_response = call_openai_chat_completion(payload=payload)
print('Raw selector output:')
print()
print(raw_selector_text)

message_obj = raw_response.get('choices', [{}])[0].get('message', {})
reasoning = message_obj.get('reasoning_content') or message_obj.get('reasoning')
if reasoning:
    print()
    print('Model reasoning trace:')
    print()
    print(reasoning)


Raw selector output:

{
  "decision": "stay",
  "active_instruction_ids": ["combination:repeat_prompt"],
  "confidence": 0.96,
  "reason": "The only instruction is currently unsatisfied (status: currently_fails) and no recommendation to switch away. Keeping it active is needed to satisfy the repeat‑prompt requirement."
}

Model reasoning trace:

We need to decide which instruction IDs to emphasize for next tokens. There's only one candidate: "combination:repeat_prompt". Currently active includes it. metadata says status currently_fails, currently_passes false, switch_away_recommended false. So instruction is unsatisfied, we should keep it (or maybe add, but it's already active). Decision: stay (keep same active set). active_instruction_ids should include it. Confidence maybe high, like 0.95. Reason: instruction currently fails, need to satisfy.


## 6) Parse + Validate Decision

This notebook uses a permissive parser that accepts `active_instruction_ids=[]`.
Validation still enforces that any selected IDs must belong to `candidate_instruction_ids`.


In [6]:
def extract_json_mapping(raw_text: str) -> dict[str, Any]:
    if not isinstance(raw_text, str):
        raise TypeError('raw_text must be a string')

    stripped = raw_text.strip()
    if not stripped:
        raise ValueError('Empty selector output')

    try:
        parsed = json.loads(stripped)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder()
    required = {'decision', 'active_instruction_ids', 'confidence'}

    for idx, ch in enumerate(stripped):
        if ch != '{':
            continue
        try:
            parsed, _ = decoder.raw_decode(stripped, idx)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict) and required.issubset(set(parsed.keys())):
            return parsed

    raise ValueError('Could not locate a valid JSON selector object')


def parse_selector_response_allow_empty(raw_text: str) -> dict[str, Any]:
    obj = extract_json_mapping(raw_text)

    decision = obj.get('decision')
    active_instruction_ids = obj.get('active_instruction_ids')
    confidence = obj.get('confidence')
    reason = obj.get('reason', '')
    metadata = obj.get('metadata', {})

    if decision not in {'stay', 'switch', 'add'}:
        raise ValueError(f"Invalid decision '{decision}'")

    if not isinstance(active_instruction_ids, list):
        raise TypeError('active_instruction_ids must be a list')
    if len(set(active_instruction_ids)) != len(active_instruction_ids):
        raise ValueError('active_instruction_ids must be unique')
    if any((not isinstance(x, str) or not x) for x in active_instruction_ids):
        raise ValueError('active_instruction_ids must contain non-empty strings')

    if isinstance(confidence, bool) or not isinstance(confidence, (int, float)):
        raise TypeError('confidence must be numeric')
    confidence = float(confidence)
    if not (0.0 <= confidence <= 1.0):
        raise ValueError('confidence must be in [0,1]')

    if not isinstance(reason, str):
        reason = str(reason)

    if metadata is None:
        metadata = {}
    if not isinstance(metadata, dict):
        metadata = {'raw_metadata': metadata}

    return {
        'decision': decision,
        'active_instruction_ids': active_instruction_ids,
        'confidence': confidence,
        'reason': reason,
        'metadata': metadata,
    }


def validate_candidate_subset(active_ids: list[str], candidate_ids: list[str]) -> None:
    if not isinstance(candidate_ids, list) or not candidate_ids:
        raise ValueError('candidate_ids must be a non-empty list')
    if any((not isinstance(x, str) or not x) for x in candidate_ids):
        raise ValueError('candidate_ids must contain non-empty strings')

    candidate_set = set(candidate_ids)
    invalid = [x for x in active_ids if x not in candidate_set]
    if invalid:
        raise ValueError(f'active_instruction_ids contain unknown ids: {invalid}')


def apply_decision(
    current_active_ids: list[str],
    decision: dict[str, Any],
    candidate_ids: list[str],
) -> list[str]:
    candidate_set = set(candidate_ids)

    if decision['decision'] == 'add':
        next_active = list(dict.fromkeys(current_active_ids + decision['active_instruction_ids']))
    elif decision['decision'] == 'switch':
        next_active = list(dict.fromkeys(decision['active_instruction_ids']))
    else:  # stay
        next_active = list(dict.fromkeys(current_active_ids or decision['active_instruction_ids']))

    invalid = [x for x in next_active if x not in candidate_set]
    if invalid:
        raise ValueError(f'Decision produced unknown ids: {invalid}')

    return next_active


decision = parse_selector_response_allow_empty(raw_selector_text)
validate_candidate_subset(decision['active_instruction_ids'], request.candidate_instruction_ids)
next_active_ids = apply_decision(initial_active_ids, decision, sample.instruction_id_list)

print('Parsed decision:')
print(json.dumps(decision, indent=2, ensure_ascii=False))
print('Next active IDs:', next_active_ids)


Parsed decision:
{
  "decision": "stay",
  "active_instruction_ids": [
    "combination:repeat_prompt"
  ],
  "confidence": 0.96,
  "reason": "The only instruction is currently unsatisfied (status: currently_fails) and no recommendation to switch away. Keeping it active is needed to satisfy the repeat‑prompt requirement.",
  "metadata": {}
}
Next active IDs: ['combination:repeat_prompt']


## 7) Optional Multi-Step Simulation

Edit `partial_generations` to test selector behavior over several boundaries.


In [9]:
partial_generations = [
    '',
    'Rewrite the following sentence in a style that is unusual: "',
    'Rewrite the following sentence in a style that is unusual: "But when the people of the land came to know that the Philistines had fled, they departed from Saul and went after David."',
    'Rewrite the following sentence in a style that is unusual: "But when the people of the land came to know that the Philistines had fled, they departed from Saul and went after David.". \n\n However, ',
]

active_ids = list(initial_active_ids)

for step, partial_text in enumerate(partial_generations, start=1):
    progress_hints = build_instruction_progress_hints(sample, partial_text)

    ctx = build_selector_context(
        sample=sample,
        current_generation=partial_text,
        active_instruction_ids=active_ids,
        generation_token_count=len(partial_text.split()),
        step_index=step,
        max_generation_chars=MAX_GENERATION_CHARS,
        extra_metadata={
            'kwargs_list': sample.kwargs_list,
            'instruction_progress_hints': progress_hints,
        },
    )
    req = SelectorRequest(
        sample_id=ctx['sample_id'],
        base_prompt=ctx['base_prompt'],
        candidate_instruction_ids=ctx['candidate_instruction_ids'],
        instruction_text_by_id=ctx['instruction_text_by_id'],
        currently_active_instruction_ids=ctx['currently_active_instruction_ids'],
        current_generation=ctx['current_generation'],
        generation_token_count=ctx['generation_token_count'],
        step_index=ctx['step_index'],
        metadata=ctx['metadata'],
    )
    pay = build_selector_payload(req, max_generation_chars=MAX_GENERATION_CHARS)

    raw_text, _ = call_openai_chat_completion(payload=pay)
    dec = parse_selector_response_allow_empty(raw_text)
    validate_candidate_subset(dec['active_instruction_ids'], req.candidate_instruction_ids)
    active_ids = apply_decision(active_ids, dec, req.candidate_instruction_ids)

    status_snapshot = {k: v.get('status', 'unknown') for k, v in progress_hints.items()}

    print(f'--- step={step} ---')
    print('partial_text:', repr(partial_text))
    print('hint_statuses:', status_snapshot)
    print('decision:', dec['decision'], dec['active_instruction_ids'], f"conf={float(dec['confidence']):.3f}")
    print('reason:', dec['reason'])
    print('next active ids:', active_ids)


--- step=1 ---
partial_text: ''
hint_statuses: {'combination:repeat_prompt': 'currently_fails'}
decision: stay ['combination:repeat_prompt'] conf=0.960
reason: The only instruction is currently unsatisfied (status: currently_fails) and no recommendation to switch away. Keeping it active is needed to satisfy the repeat‑prompt requirement.
next active ids: ['combination:repeat_prompt']
--- step=2 ---
partial_text: 'Rewrite the following sentence in a style that is unusual: "'
hint_statuses: {'combination:repeat_prompt': 'currently_fails'}
decision: stay ['combination:repeat_prompt'] conf=0.960
reason: The only instruction is currently unsatisfied (status: currently_fails) and no recommendation to switch away. Keeping it active is needed to ensure the model repeats the request word‑for‑word before providing the answer.
next active ids: ['combination:repeat_prompt']
--- step=3 ---
partial_text: 'Rewrite the following sentence in a style that is unusual: "But when the people of the land cam

## Notes

- This notebook uses your endpoint as a selector-only model.
- It intentionally allows empty active sets (`active_instruction_ids=[]`) for experimentation.
- Progress hints are computed generically via the IFEval checker registry with no hardcoded per-instruction logic.
- The current production dynamic runner still assumes non-empty active sets; if needed, we can patch runner/controller to support empty sets end-to-end.
